## Similarity Approach — Alarm Clusters

**Approach**: Build a context lookup table from historical alarm clusters, then use 4-component similarity to recommend control actions for new alarm clusters.

### Key Differences from Episode-Based Approach
- Uses **alarm clusters** (groups of alarms with ≤30 min gap) instead of individual SSD alarm episodes
- Context window starts at `cluster_start - 30 min` (instead of SSD deviation start)
- Control actions come pre-extracted from the clustered Excel file
- Excludes trip/shutdown clusters (PV ≤ 0 for ≥ 30 min)

### 4 Similarity Components
| Component | Distance Metric | Weight | What it captures |
|-----------|----------------|--------|------------------|
| **NormPos** (Normalized Position) | Euclidean | 0.25 | Where the plant is right now relative to operating limits |
| **NormROC** (Normalized Rate of Change) | Euclidean | 0.30 | How fast each tag is changing relative to its range |
| **Alarm Proximity** | Absolute difference | 0.20 | How close 03LIC_1071 is to the alarm threshold |
| **Weighted Direction Match** | Weighted agreement | 0.25 | Are tags moving in the same directions (weighted by magnitude) |

### Target Tags for Actions
- **03LIC_1071** (target level controller)
- **03LIC_1016** (related level controller)
- **03PIC_1013** (pressure controller)

In [12]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load PV/OP time series data
pv_op_data_df = pd.read_parquet('/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet')
pv_op_data_df.sort_index(inplace=True)

# Load alarm clusters and control actions from Excel
alarms_df = pd.read_excel('/home/h604827/ControlActions/DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx', sheet_name=0)
actions_df = pd.read_excel('/home/h604827/ControlActions/DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx', sheet_name=1)
actions_df['VT_Start'] = pd.to_datetime(actions_df['VT_Start'])

# Load operating limits
operating_limits_df = pd.read_csv('/home/h604827/ControlActions/DATA/operating_limits.csv')

# Constants
ALARM_THRESHOLD = 28.75
TARGET_TAG = '03LIC_1071'
TARGET_SOURCES = ['03LIC_1071', '03LIC_1016', '03PIC_1013']
CONTEXT_WINDOW_BEFORE = 30  # minutes before cluster_start for context window

print(f'PV/OP data: {pv_op_data_df.shape[0]:,} rows, {pv_op_data_df.index.min()} to {pv_op_data_df.index.max()}')
print(f'Alarm clusters: {alarms_df["cluster_id"].nunique()} clusters, {len(alarms_df)} individual alarms')
print(f'Control actions: {len(actions_df)} actions across {actions_df["cluster_id"].nunique()} clusters')
print(f'Operating limits: {len(operating_limits_df)} tags')

PV/OP data: 1,718,039 rows, 2022-01-03 22:45:00 to 2025-06-23 20:44:00
Alarm clusters: 539 clusters, 1379 individual alarms
Control actions: 16094 actions across 430 clusters
Operating limits: 40 tags


In [13]:
# Build cluster-level summary and exclude trip/shutdown clusters (PV <= 0 for >= 30 min)
clusters_agg = alarms_df.groupby('cluster_id').agg(
    cluster_start=('cluster_start_time', 'first'),
    cluster_end=('cluster_end_time', 'first'),
    cluster_type=('cluster_type', 'first'),
    n_alarms=('cluster_total_alarms', 'first')
).sort_values('cluster_start')

# Identify trip/shutdown clusters
trip_clusters = set()
for cid, row in clusters_agg.iterrows():
    ws = row['cluster_start'] - pd.Timedelta(minutes=30)
    we = row['cluster_end'] + pd.Timedelta(minutes=30)
    window = pv_op_data_df.loc[ws:we, '03LIC_1071.PV']
    if window.empty:
        trip_clusters.add(cid)
        continue
    below_zero = (window <= 0).astype(int)
    if below_zero.sum() == 0:
        continue
    groups = (below_zero != below_zero.shift()).cumsum()
    for _, grp in window.groupby(groups):
        if (grp <= 0).all() and len(grp) > 1:
            dur_min = (grp.index[-1] - grp.index[0]).total_seconds() / 60
            if dur_min >= 30:
                trip_clusters.add(cid)
                break

# Filter out trip clusters
valid_cluster_ids = sorted(set(clusters_agg.index) - trip_clusters)
clusters_valid = clusters_agg.loc[valid_cluster_ids].copy()

print(f'Total clusters: {len(clusters_agg)}')
print(f'Trip/shutdown clusters excluded: {len(trip_clusters)} → {sorted(trip_clusters)}')
print(f'Valid clusters: {len(clusters_valid)}')

# Get actions for target tags only (SP/OP only, exclude MODE etc.)
target_actions_df = actions_df[
    (actions_df['Source'].isin(TARGET_SOURCES)) &
    (actions_df['Description'].isin(['SP', 'OP']))
].copy()

# Filter out actions from trip clusters
target_actions_df = target_actions_df[target_actions_df['cluster_id'].isin(valid_cluster_ids)].copy()

print(f'\nTarget tag SP/OP actions (valid clusters): {len(target_actions_df)}')
print(f'Actions by source:')
print(target_actions_df['Source'].value_counts().to_string())
print(f'\nClusters with target actions: {target_actions_df["cluster_id"].nunique()}')

Total clusters: 539
Trip/shutdown clusters excluded: 9 → [80, 214, 216, 432, 434, 475, 476, 481, 534]
Valid clusters: 530

Target tag SP/OP actions (valid clusters): 2886
Actions by source:
Source
03PIC_1013    1778
03LIC_1071     760
03LIC_1016     348

Clusters with target actions: 175


### Train/Test Split

- **Training**: All valid clusters from 2022–2024
- **Test**: 50 random clusters from 2025 (that have target tag actions)

In [14]:
# Train/Test split by year
# Clusters with target actions
clusters_with_actions = sorted(target_actions_df['cluster_id'].unique())
clusters_with_actions_df = clusters_valid.loc[clusters_valid.index.isin(clusters_with_actions)].copy()

# 2025 clusters for testing
clusters_2025 = clusters_with_actions_df[
    clusters_with_actions_df['cluster_start'] >= pd.Timestamp('2025-01-01')
]

# Sample 50 test clusters (or all if fewer)
np.random.seed(42)
sample_n = min(50, len(clusters_2025))
test_cluster_ids = sorted(np.random.choice(clusters_2025.index, size=sample_n, replace=False))

# Training: all clusters with actions, excluding test clusters
train_cluster_ids = sorted(set(clusters_with_actions) - set(test_cluster_ids))

print(f'Clusters with target tag actions: {len(clusters_with_actions)}')
print(f'  2025 clusters: {len(clusters_2025)}')
print(f'  Test clusters (2025): {len(test_cluster_ids)}')
print(f'  Training clusters: {len(train_cluster_ids)}')

Clusters with target tag actions: 175
  2025 clusters: 47
  Test clusters (2025): 47
  Training clusters: 128


### Helper Functions

- **Operating limits lookup**: Normalized position within operating range
- **PV value lookup**: Get PV at a specific timestamp (with nearest-value fallback)
- **Sub-minute action merging**: Collapse multiple actions on the same tag within the same minute into one net action
- **Context builder**: Build context vector for a given timestamp

In [15]:
# Build operating limits lookup dict
op_limits = {}
for _, row in operating_limits_df.iterrows():
    tag_name = row['TAG_NAME']
    tag_base = tag_name.replace('.PV', '').replace('.OP', '')
    lower = row['LOWER_LIMIT']
    upper = row['UPPER_LIMIT']
    op_range = upper - lower
    if op_range > 0:
        op_limits[tag_base] = {'lower': lower, 'upper': upper, 'range': op_range}

target_upper = op_limits[TARGET_TAG]['upper']
print(f'Operating limits loaded for {len(op_limits)} tags')
print(f'{TARGET_TAG}: lower={op_limits[TARGET_TAG]["lower"]:.2f}, upper={target_upper:.2f}, alarm={ALARM_THRESHOLD}')

# PV tag columns for context
context_pv_tags = [col for col in pv_op_data_df.columns if col.endswith('.PV')]
print(f'Context PV tags: {len(context_pv_tags)}')

Operating limits loaded for 26 tags
03LIC_1071: lower=35.25, upper=42.41, alarm=28.75
Context PV tags: 28


In [16]:
def get_pv_at_timestamp(timestamp, pv_tag):
    """Get PV value at or nearest to the given timestamp."""
    try:
        if hasattr(timestamp, 'tzinfo') and timestamp.tzinfo is not None:
            timestamp = timestamp.tz_localize(None)
        if timestamp in pv_op_data_df.index:
            return pv_op_data_df.loc[timestamp, pv_tag]
        idx = pv_op_data_df.index.get_indexer([timestamp], method='ffill')[0]
        if 0 <= idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        idx = pv_op_data_df.index.get_indexer([timestamp], method='bfill')[0]
        if 0 <= idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        return np.nan
    except Exception:
        return np.nan


def merge_subminute_actions(actions_in):
    """
    Merge actions on the same tag within the same minute into a single composite action.
    PV/OP data is minute-wise, so sub-minute actions all get the same context.
    Collapses them: PrevValue = first action's PrevValue, Value = last action's Value.
    """
    if len(actions_in) == 0:
        return pd.DataFrame(columns=['Source', 'VT_Start', 'Value', 'PrevValue',
                                      'Value_num', 'PrevValue_num', 'Description',
                                      'ConditionName', 'num_raw_actions', 'magnitude'])

    actions = actions_in.copy()
    actions['Value_num'] = pd.to_numeric(actions['Value'], errors='coerce')
    actions['PrevValue_num'] = pd.to_numeric(actions['PrevValue'], errors='coerce')
    actions = actions.dropna(subset=['Value_num', 'PrevValue_num'])

    if len(actions) == 0:
        return pd.DataFrame(columns=['Source', 'VT_Start', 'Value', 'PrevValue',
                                      'Value_num', 'PrevValue_num', 'Description',
                                      'ConditionName', 'num_raw_actions', 'magnitude'])

    actions = actions.drop_duplicates(subset=['VT_Start', 'Source', 'Value'])
    actions['minute_floor'] = actions['VT_Start'].dt.floor('min')
    actions = actions.sort_values(['Source', 'VT_Start'])

    merged_records = []
    for (source, minute), group in actions.groupby(['Source', 'minute_floor']):
        group_sorted = group.sort_values('VT_Start')
        first_row = group_sorted.iloc[0]
        last_row = group_sorted.iloc[-1]
        merged_records.append({
            'Source': source,
            'VT_Start': minute,
            'Value': last_row['Value_num'],
            'PrevValue': first_row['PrevValue_num'],
            'Value_num': last_row['Value_num'],
            'PrevValue_num': first_row['PrevValue_num'],
            'ConditionName': 'CHANGE',
            'Description': group_sorted['Description'].mode().iloc[0] if 'Description' in group_sorted.columns else None,
            'num_raw_actions': len(group_sorted),
        })

    result = pd.DataFrame(merged_records)
    result['magnitude'] = result['Value_num'] - result['PrevValue_num']
    return result

print('Helper functions defined: get_pv_at_timestamp(), merge_subminute_actions()')

Helper functions defined: get_pv_at_timestamp(), merge_subminute_actions()


In [17]:
# Typical cluster duration for time_progress normalization (from training clusters)
train_durations = []
for cid in train_cluster_ids:
    row = clusters_valid.loc[cid]
    dur = (row['cluster_end'] - row['cluster_start']).total_seconds() / 60.0
    train_durations.append(dur)
TYPICAL_CLUSTER_DURATION_MINUTES = np.median(train_durations)
print(f'Typical cluster duration (median from training): {TYPICAL_CLUSTER_DURATION_MINUTES:.1f} minutes')


def build_context(context_start, action_timestamp, pv_tags):
    """
    Build context features at a specific timestamp.
    
    Context features per PV tag (28 tags):
      - {tag}_norm_pos: (PV_current - Lower) / Range → position within operating limits
      - {tag}_norm_roc: (PV_current - PV_start) / Range → change as fraction of range
      - {tag}_roc_direction: 1 if rising, 0 if falling
    
    Global features:
      - alarm_proximity: (PV_1071 - 28.75) / (Upper_1071 - 28.75) → 0 at alarm, 1 at upper
      - time_progress: minutes_since_context_start / typical_cluster_duration
    """
    context = {}

    for pv_tag in pv_tags:
        tag_base = pv_tag.replace('.PV', '')
        pv_at_start = get_pv_at_timestamp(context_start, pv_tag)
        pv_at_action = get_pv_at_timestamp(action_timestamp, pv_tag)
        limits = op_limits.get(tag_base, None)

        if limits and pd.notna(pv_at_action):
            norm_pos = (pv_at_action - limits['lower']) / limits['range']
        else:
            norm_pos = np.nan

        if limits and pd.notna(pv_at_start) and pd.notna(pv_at_action):
            norm_roc = (pv_at_action - pv_at_start) / limits['range']
        else:
            norm_roc = np.nan

        roc_direction = (1 if norm_roc >= 0 else 0) if pd.notna(norm_roc) else np.nan

        context[f'{tag_base}_norm_pos'] = norm_pos
        context[f'{tag_base}_norm_roc'] = norm_roc
        context[f'{tag_base}_roc_direction'] = roc_direction

    # Alarm proximity
    pv_1071 = get_pv_at_timestamp(action_timestamp, '03LIC_1071.PV')
    if pd.notna(pv_1071):
        context['alarm_proximity'] = (pv_1071 - ALARM_THRESHOLD) / (target_upper - ALARM_THRESHOLD)
    else:
        context['alarm_proximity'] = np.nan

    # Time progress
    time_delta = (action_timestamp - context_start).total_seconds() / 60.0
    context['time_progress'] = time_delta / TYPICAL_CLUSTER_DURATION_MINUTES

    return context

print('Context builder function defined: build_context()')

Typical cluster duration (median from training): 17.5 minutes
Context builder function defined: build_context()


### Build Training Context Lookup Table

For each training cluster, extract SP/OP actions on target tags, merge sub-minute actions, and build context at each action timestamp.

Context window starts at `cluster_start - 30 min`.

In [18]:
# Build training context lookup table
context_records = []
raw_action_count = 0
merged_action_count = 0

print(f'Processing {len(train_cluster_ids)} training clusters...')
for cid in tqdm(train_cluster_ids, desc='Training clusters'):
    cluster = clusters_valid.loc[cid]
    cluster_start = cluster['cluster_start']
    cluster_end = cluster['cluster_end']
    context_start = cluster_start - pd.Timedelta(minutes=CONTEXT_WINDOW_BEFORE)

    # Get target tag SP/OP actions for this cluster
    cluster_actions_raw = target_actions_df[target_actions_df['cluster_id'] == cid].copy()
    if len(cluster_actions_raw) == 0:
        continue

    raw_action_count += len(cluster_actions_raw)

    # Merge sub-minute actions
    cluster_actions_merged = merge_subminute_actions(cluster_actions_raw)
    if len(cluster_actions_merged) == 0:
        continue

    merged_action_count += len(cluster_actions_merged)

    # Build context for each merged action
    for _, action in cluster_actions_merged.iterrows():
        action_ts = action['VT_Start']
        ctx = build_context(context_start, action_ts, context_pv_tags)

        # Add metadata
        ctx['cluster_id'] = cid
        ctx['cluster_start'] = cluster_start
        ctx['cluster_end'] = cluster_end
        ctx['context_start'] = context_start
        ctx['action_timestamp'] = action_ts
        ctx['action_source'] = action['Source']
        ctx['action_type'] = action['Description']
        ctx['action_value'] = action['Value']
        ctx['action_prev_value'] = action['PrevValue']
        ctx['action_magnitude'] = action['magnitude']
        ctx['action_direction'] = 1 if action['magnitude'] > 0 else 0
        ctx['num_raw_actions'] = action['num_raw_actions']

        context_records.append(ctx)

print(f'\nRaw actions: {raw_action_count}')
print(f'After sub-minute merging: {merged_action_count}')
print(f'Reduction: {raw_action_count - merged_action_count} merged away '
      f'({(1 - merged_action_count/raw_action_count)*100:.1f}%)')
print(f'Total context records: {len(context_records)}')

Processing 128 training clusters...


Training clusters: 100%|██████████| 128/128 [00:51<00:00,  2.47it/s]


Raw actions: 2316
After sub-minute merging: 911
Reduction: 1405 merged away (60.7%)
Total context records: 911


In [19]:
# Convert to DataFrame and clean
context_df = pd.DataFrame(context_records)

# Remove NaN magnitudes and duplicates
context_df_clean = context_df[context_df['action_magnitude'].notna()].copy()
context_df_clean = context_df_clean.drop_duplicates(
    subset=['cluster_id', 'action_timestamp', 'action_source']
)

print(f'Context DataFrame: {context_df_clean.shape}')
print(f'Unique training clusters represented: {context_df_clean["cluster_id"].nunique()}')
print(f'\nActions per cluster:')
apc = context_df_clean.groupby('cluster_id').size()
print(f'  Min: {apc.min()}, Max: {apc.max()}, Mean: {apc.mean():.1f}, Median: {apc.median():.0f}')
print(f'\nActions by source:')
print(context_df_clean['action_source'].value_counts().to_string())
print(f'\nAction direction: {context_df_clean["action_direction"].value_counts().to_string()}')

# Feature column sets
norm_pos_cols = [c for c in context_df_clean.columns if '_norm_pos' in c]
norm_roc_cols = [c for c in context_df_clean.columns if '_norm_roc' in c]
dir_cols = [c for c in context_df_clean.columns if '_roc_direction' in c]

# Keep only tags with valid operating limits
valid_tags = set(op_limits.keys())
norm_pos_cols = [c for c in norm_pos_cols if c.replace('_norm_pos', '') in valid_tags]
norm_roc_cols = [c for c in norm_roc_cols if c.replace('_norm_roc', '') in valid_tags]
dir_cols = [c for c in dir_cols if c.replace('_roc_direction', '') in valid_tags]

print(f'\nFeature columns: {len(norm_pos_cols)} NormPos, {len(norm_roc_cols)} NormROC, {len(dir_cols)} Direction')
print(f'Global features: alarm_proximity, time_progress')

Context DataFrame: (911, 98)
Unique training clusters represented: 128

Actions per cluster:
  Min: 1, Max: 90, Mean: 7.1, Median: 3

Actions by source:
action_source
03PIC_1013    442
03LIC_1071    289
03LIC_1016    180

Action direction: action_direction
1    481
0    430

Feature columns: 26 NormPos, 26 NormROC, 26 Direction
Global features: alarm_proximity, time_progress


In [20]:
# Save training context
output_dir = '/home/h604827/ControlActions/RESULTS/similarity_test_results/alarm_clusters_v1'
os.makedirs(output_dir, exist_ok=True)

context_df_clean.to_csv(f'{output_dir}/similarity_context_training.csv', index=False)

metadata = {
    'train_cluster_ids': train_cluster_ids,
    'test_cluster_ids': test_cluster_ids,
    'trip_clusters_excluded': sorted(trip_clusters),
    'total_valid_clusters': len(clusters_valid),
    'training_clusters_count': len(train_cluster_ids),
    'test_clusters_count': len(test_cluster_ids),
    'context_window_before_min': CONTEXT_WINDOW_BEFORE,
    'typical_cluster_duration_min': TYPICAL_CLUSTER_DURATION_MINUTES,
}
with open(f'{output_dir}/context_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f'Training context saved to {output_dir}/')
print(f'  Context shape: {context_df_clean.shape}')

Training context saved to /home/h604827/ControlActions/RESULTS/similarity_test_results/alarm_clusters_v1/
  Context shape: (911, 98)


### 4-Component Similarity Function

Vectorized computation — no per-row Python loop.

In [21]:
def calculate_weighted_similarity(runtime_context, historical_df,
                                   norm_pos_cols, norm_roc_cols, dir_cols,
                                   w_norm_pos=0.25, w_norm_roc=0.30,
                                   w_alarm_prox=0.20, w_dir=0.25):
    """
    Calculate 4-component similarity between runtime context and all historical contexts.
    
    Components:
    1. NormPos Similarity (Euclidean → similarity) — WHERE each tag is
    2. NormROC Similarity (Euclidean → similarity) — HOW FAST each tag is changing
    3. Alarm Proximity Similarity (abs diff → similarity) — HOW URGENT
    4. Weighted Direction Match — ARE tags moving the same way (weighted by magnitude)
    """
    n_tags = len(norm_pos_cols)

    # Runtime vectors
    runtime_norm_pos = np.nan_to_num([runtime_context.get(c, 0) for c in norm_pos_cols])
    runtime_norm_roc = np.nan_to_num([runtime_context.get(c, 0) for c in norm_roc_cols])
    runtime_dir = np.nan_to_num([runtime_context.get(c, 0) for c in dir_cols])
    runtime_alarm_prox = runtime_context.get('alarm_proximity', 0.5)
    if pd.isna(runtime_alarm_prox):
        runtime_alarm_prox = 0.5

    # Historical arrays
    hist_norm_pos = historical_df[norm_pos_cols].fillna(0).values
    hist_norm_roc = historical_df[norm_roc_cols].fillna(0).values
    hist_dir = historical_df[dir_cols].fillna(0).values
    hist_alarm_prox = historical_df['alarm_proximity'].fillna(0.5).values

    # Component 1: NormPos similarity
    norm_pos_dists = np.sqrt(np.sum((hist_norm_pos - runtime_norm_pos) ** 2, axis=1))
    norm_pos_sim = 1.0 - (norm_pos_dists / np.sqrt(n_tags))

    # Component 2: NormROC similarity
    norm_roc_dists = np.sqrt(np.sum((hist_norm_roc - runtime_norm_roc) ** 2, axis=1))
    norm_roc_sim = 1.0 - np.clip(norm_roc_dists / np.sqrt(n_tags * 4), 0, 1)

    # Component 3: Alarm proximity similarity
    alarm_prox_sim = 1.0 - np.clip(np.abs(hist_alarm_prox - runtime_alarm_prox), 0, 1)

    # Component 4: Weighted direction match
    abs_runtime_roc = np.abs(runtime_norm_roc)
    abs_hist_roc = np.abs(hist_norm_roc)
    movement_weights = np.maximum(abs_runtime_roc, abs_hist_roc)
    dir_matches = (hist_dir == runtime_dir).astype(float)
    weight_sums = movement_weights.sum(axis=1)
    safe_weight_sums = np.where(weight_sums > 0, weight_sums, 1.0)
    weighted_dir_match = np.where(
        weight_sums > 0,
        (dir_matches * movement_weights).sum(axis=1) / safe_weight_sums,
        0.5
    )

    # Combined
    total_sim = (w_norm_pos * norm_pos_sim +
                 w_norm_roc * norm_roc_sim +
                 w_alarm_prox * alarm_prox_sim +
                 w_dir * weighted_dir_match)

    results = pd.DataFrame({
        'hist_index': historical_df.index,
        'total_similarity': total_sim,
        'norm_pos_similarity': norm_pos_sim,
        'norm_roc_similarity': norm_roc_sim,
        'alarm_prox_similarity': alarm_prox_sim,
        'weighted_dir_match': weighted_dir_match,
        'action_source': historical_df['action_source'].values,
        'action_type': historical_df['action_type'].values,
        'action_direction': historical_df['action_direction'].values,
        'action_magnitude': historical_df['action_magnitude'].values,
        'cluster_id': historical_df['cluster_id'].values,
        'num_raw_actions': historical_df['num_raw_actions'].values,
    })

    return results.sort_values('total_similarity', ascending=False)

print('4-component similarity function defined (vectorized)')

4-component similarity function defined (vectorized)


### Run Similarity on Test Clusters (2025)

For each test cluster:
1. Get the actual target-tag SP/OP actions (with sub-minute merging)
2. At each action timestamp, build context and find top-3 similar historical contexts
3. Compare recommended vs actual: tag, type (SP/OP), direction, magnitude
4. Generate interactive HTML visualization per cluster
5. Save Excel report with actual vs recommended actions

In [22]:
# Run similarity matching on test clusters
magnitude_tolerance = 3  # abs(diff) <= this counts as magnitude match

target_pv_tags_vis = ['03LIC_1071.PV', '03LIC_1016.PV', '03PIC_1013.PV']
target_pv_colors = {'03LIC_1071.PV': 'blue', '03LIC_1016.PV': 'green', '03PIC_1013.PV': 'orange'}

all_episodes_summary = []
excel_rows_by_cluster = {}

overall_counts = {
    'total_actual': 0, 'tag_matches': 0, 'type_matches': 0,
    'dir_matches': 0, 'mag_matches': 0, 'all_matches': 0
}

print(f'=== 4-Component Similarity + Sub-Minute Merging (Alarm Clusters) ===')
print(f'Training lookup: {len(context_df_clean)} entries from {context_df_clean["cluster_id"].nunique()} clusters')
print(f'Test clusters: {len(test_cluster_ids)}')
print(f'Output: {output_dir}/\n')

for test_cid in tqdm(test_cluster_ids, desc='Test clusters'):
    cluster = clusters_valid.loc[test_cid]
    cluster_start = cluster['cluster_start']
    cluster_end = cluster['cluster_end']
    context_start = cluster_start - pd.Timedelta(minutes=CONTEXT_WINDOW_BEFORE)

    # Get actual actions for this cluster
    cluster_actions_raw = target_actions_df[target_actions_df['cluster_id'] == test_cid].copy()
    if len(cluster_actions_raw) == 0:
        continue

    actual_merged = merge_subminute_actions(cluster_actions_raw)
    if len(actual_merged) == 0:
        continue

    actual_merged['action_direction'] = (actual_merged['magnitude'] > 0).astype(int)
    actual_merged['action_type'] = actual_merged['Description']

    # Run similarity at each merged action timestamp
    cluster_results = []
    for _, act in actual_merged.iterrows():
        current_time = pd.to_datetime(act['VT_Start'])
        runtime_ctx = build_context(context_start, current_time, context_pv_tags)
        sim_results = calculate_weighted_similarity(
            runtime_ctx, context_df_clean, norm_pos_cols, norm_roc_cols, dir_cols
        )

        for rank, (_, match) in enumerate(sim_results.head(3).iterrows(), 1):
            cluster_results.append({
                'cluster_id': test_cid,
                'action_time': current_time,
                'rank': rank,
                'similarity': match['total_similarity'],
                'norm_pos_similarity': match['norm_pos_similarity'],
                'norm_roc_similarity': match['norm_roc_similarity'],
                'alarm_prox_similarity': match['alarm_prox_similarity'],
                'weighted_dir_match': match['weighted_dir_match'],
                'recommended_action_source': match['action_source'],
                'recommended_action_type': match['action_type'],
                'recommended_action_direction': match['action_direction'],
                'recommended_action_magnitude': match['action_magnitude'],
                'recommended_num_raw_actions': int(match['num_raw_actions']),
                'matched_cluster_id': match['cluster_id'],
                'actual_action_source': act['Source'],
                'actual_action_type': act['action_type'],
                'actual_action_direction': int(act['action_direction']),
                'actual_action_magnitude': float(act['magnitude']),
                'actual_num_raw_actions': int(act['num_raw_actions']),
                'alarm_proximity': runtime_ctx['alarm_proximity'],
                'time_progress': runtime_ctx['time_progress'],
                '1071_norm_roc': runtime_ctx.get('03LIC_1071_norm_roc', 0),
            })

    if len(cluster_results) == 0:
        continue

    results_df = pd.DataFrame(cluster_results)
    top_recs = results_df[results_df['rank'] == 1].copy().sort_values('action_time')

    # ── Visualization ──
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=[
            f'Cluster {test_cid}: All PV Tags (Normalized to Operating Limits)',
            'Recommended Action Source at Action Times',
            'Similarity Components at Action Times'
        ],
        vertical_spacing=0.08, row_heights=[0.40, 0.25, 0.35]
    )

    pv_start = context_start - pd.Timedelta(minutes=10)
    pv_end = cluster_end + pd.Timedelta(minutes=70)
    actual_tags_str = ', '.join(sorted(actual_merged['Source'].unique()))

    # Row 1: PV tags normalized to operating limits
    for pv_tag in context_pv_tags:
        tag_base = pv_tag.replace('.PV', '')
        limits = op_limits.get(tag_base, None)
        if limits is None:
            continue
        pv_window = pv_op_data_df.loc[pv_start:pv_end, pv_tag]
        if pv_window.empty:
            continue
        norm_vals = (pv_window.values - limits['lower']) / limits['range']
        is_target = pv_tag in target_pv_tags_vis
        fig.add_trace(
            go.Scatter(
                x=pv_window.index, y=norm_vals, mode='lines', name=pv_tag,
                line=dict(color=target_pv_colors.get(pv_tag, None),
                          width=3 if pv_tag == '03LIC_1071.PV' else (2 if is_target else 1)),
                visible=True if is_target else 'legendonly',
                hovertemplate=f'{pv_tag}<br>Norm: %{{y:.3f}}<br>Raw: %{{customdata:.2f}}<br>Time: %{{x}}',
                customdata=pv_window.values
            ), row=1, col=1
        )

    # Alarm threshold line
    if TARGET_TAG in op_limits:
        alarm_norm = (ALARM_THRESHOLD - op_limits[TARGET_TAG]['lower']) / op_limits[TARGET_TAG]['range']
        fig.add_hline(y=alarm_norm, line_dash='dash', line_color='red',
                      annotation_text=f'1071 Alarm ({ALARM_THRESHOLD})', row=1, col=1)

    # Shading: context window (orange), alarm cluster (red)
    fig.add_vrect(x0=context_start, x1=cluster_start, fillcolor='orange', opacity=0.1, line_width=0, row=1, col=1)
    fig.add_vrect(x0=cluster_start, x1=cluster_end, fillcolor='red', opacity=0.2, line_width=0, row=1, col=1)

    # Action markers on PV plot
    action_times = pd.to_datetime(actual_merged['VT_Start'])
    if TARGET_TAG in op_limits:
        action_pv_raw = [get_pv_at_timestamp(t, '03LIC_1071.PV') for t in action_times]
        action_pv_norm = [
            (v - op_limits[TARGET_TAG]['lower']) / op_limits[TARGET_TAG]['range'] if pd.notna(v) else np.nan
            for v in action_pv_raw
        ]
    else:
        action_pv_norm = [np.nan] * len(action_times)

    fig.add_trace(
        go.Scatter(
            x=action_times, y=action_pv_norm, mode='markers', name='Actual Actions',
            marker=dict(symbol='triangle-up', size=20, color='red'),
            text=[f"{t} Net: {m:+.1f} ({n} raw)" for t, m, n in
                  zip(actual_merged['action_type'], actual_merged['magnitude'], actual_merged['num_raw_actions'])],
            hovertemplate='%{text}<br>Time: %{x}'
        ), row=1, col=1
    )

    # Row 2: Recommended source
    source_map = {'03LIC_1071': 0, '03LIC_1016': 1, '03PIC_1013': 2}
    top_recs['source_numeric'] = top_recs['recommended_action_source'].map(source_map)
    colors = ['green' if d == 1 else 'red' for d in top_recs['recommended_action_direction']]
    fig.add_trace(
        go.Scatter(
            x=top_recs['action_time'], y=top_recs['source_numeric'],
            mode='markers', marker=dict(size=8, color=colors),
            name='Recommended (green=up, red=down)',
            text=[f"Tag: {s}<br>Type: {t}<br>Dir: {'up' if d==1 else 'down'}<br>Mag: {mag:+.1f} ({n} steps)<br>Sim: {sim:.3f}"
                  for s, t, d, mag, n, sim in zip(top_recs['recommended_action_source'],
                                                   top_recs['recommended_action_type'],
                                                   top_recs['recommended_action_direction'],
                                                   top_recs['recommended_action_magnitude'],
                                                   top_recs['recommended_num_raw_actions'],
                                                   top_recs['similarity'])],
            hoverinfo='text'
        ), row=2, col=1
    )
    fig.add_vrect(x0=cluster_start, x1=cluster_end, fillcolor='red', opacity=0.2, line_width=0, row=2, col=1)

    # Row 3: Similarity components
    fig.add_trace(go.Scatter(x=top_recs['action_time'], y=top_recs['similarity'],
                  mode='markers+lines', name='Total Similarity', line=dict(color='purple', width=2)), row=3, col=1)
    fig.add_trace(go.Scatter(x=top_recs['action_time'], y=top_recs['norm_pos_similarity'],
                  mode='markers+lines', name='NormPos Sim', line=dict(color='blue', dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=top_recs['action_time'], y=top_recs['norm_roc_similarity'],
                  mode='markers+lines', name='NormROC Sim', line=dict(color='green', dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=top_recs['action_time'], y=top_recs['alarm_prox_similarity'],
                  mode='markers+lines', name='Alarm Prox Sim', line=dict(color='red', dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=top_recs['action_time'], y=top_recs['weighted_dir_match'],
                  mode='markers+lines', name='Weighted Dir Match', line=dict(color='orange', dash='dot')), row=3, col=1)
    fig.add_vrect(x0=cluster_start, x1=cluster_end, fillcolor='red', opacity=0.2, line_width=0, row=3, col=1)

    fig.update_layout(
        height=1100, showlegend=True, hovermode='closest',
        title_text=f'Cluster {test_cid} - Similarity + Sub-Minute Merging<br>'
                   f'<sub>Actual Tags: {actual_tags_str} | Actions: {len(actual_merged)} merged from {actual_merged["num_raw_actions"].sum()} raw</sub>'
    )
    fig.update_xaxes(range=[pv_start, pv_end], row=1, col=1)
    fig.update_xaxes(range=[pv_start, pv_end], row=2, col=1)
    fig.update_xaxes(range=[pv_start, pv_end], row=3, col=1)
    fig.update_yaxes(title_text='Normalized Position (0=Lower, 1=Upper)', row=1, col=1)
    fig.update_yaxes(title_text='Tag', ticktext=['1071', '1016', '1013'], tickvals=[0, 1, 2], row=2, col=1)
    fig.update_yaxes(title_text='Similarity Score', row=3, col=1)

    fig.write_html(f'{output_dir}/cluster_{test_cid}_visualization.html')

    # ── Excel rows + metrics ──
    rows = []
    tag_matches = type_matches = dir_matches = mag_matches = all_matches = 0
    total_actual = len(actual_merged)

    for i, (_, act) in enumerate(actual_merged.iterrows(), start=1):
        act_time = pd.to_datetime(act['VT_Start'])
        rows.append({
            'cluster_id': test_cid, 'action_group': i, 'row_type': 'actual',
            'action_time': act_time, 'source': act['Source'], 'action_type': act['action_type'],
            'direction': int(act['action_direction']), 'magnitude': float(act['magnitude']),
            'prev_value': act['PrevValue'], 'value': act['Value'],
            'num_raw_actions': int(act['num_raw_actions']),
            'similarity': np.nan, 'matched_cluster_id': np.nan,
            'tag_match': np.nan, 'type_match': np.nan, 'direction_match': np.nan, 'magnitude_match': np.nan,
        })

        recs = results_df[results_df['action_time'] == act_time].sort_values('rank')
        for _, rec in recs.iterrows():
            tag_ok = rec['recommended_action_source'] == act['Source']
            type_ok = rec['recommended_action_type'] == act['action_type']
            dir_ok = int(rec['recommended_action_direction']) == int(act['action_direction'])
            mag_ok = pd.notna(rec['recommended_action_magnitude']) and abs(rec['recommended_action_magnitude'] - act['magnitude']) <= magnitude_tolerance

            rows.append({
                'cluster_id': test_cid, 'action_group': i, 'row_type': f'recommended_rank{int(rec["rank"])}',
                'action_time': act_time, 'source': rec['recommended_action_source'],
                'action_type': rec['recommended_action_type'],
                'direction': int(rec['recommended_action_direction']),
                'magnitude': rec['recommended_action_magnitude'],
                'prev_value': np.nan, 'value': np.nan,
                'num_raw_actions': int(rec['recommended_num_raw_actions']),
                'similarity': rec['similarity'], 'matched_cluster_id': rec['matched_cluster_id'],
                'tag_match': 1 if tag_ok else 0, 'type_match': 1 if type_ok else 0,
                'direction_match': 1 if dir_ok else 0, 'magnitude_match': 1 if mag_ok else 0,
            })

        # Top-1 metrics
        top1 = top_recs[top_recs['action_time'] == act_time]
        if len(top1) > 0:
            top1 = top1.iloc[0]
            t_ok = top1['recommended_action_source'] == act['Source']
            tp_ok = top1['recommended_action_type'] == act['action_type']
            d_ok = int(top1['recommended_action_direction']) == int(act['action_direction'])
            m_ok = pd.notna(top1['recommended_action_magnitude']) and abs(top1['recommended_action_magnitude'] - act['magnitude']) <= magnitude_tolerance
            if t_ok: tag_matches += 1
            if tp_ok: type_matches += 1
            if d_ok: dir_matches += 1
            if m_ok: mag_matches += 1
            if t_ok and tp_ok and d_ok and m_ok: all_matches += 1

    excel_rows_by_cluster[test_cid] = pd.DataFrame(rows)

    all_episodes_summary.append({
        'cluster_id': test_cid,
        'cluster_start': str(cluster_start),
        'cluster_end': str(cluster_end),
        'actual_actions_count': total_actual,
        'tag_match_accuracy': tag_matches / total_actual if total_actual > 0 else None,
        'type_match_accuracy': type_matches / total_actual if total_actual > 0 else None,
        'direction_match_accuracy': dir_matches / total_actual if total_actual > 0 else None,
        'magnitude_match_accuracy': mag_matches / total_actual if total_actual > 0 else None,
        'all_match_accuracy': all_matches / total_actual if total_actual > 0 else None,
    })
    overall_counts['total_actual'] += total_actual
    overall_counts['tag_matches'] += tag_matches
    overall_counts['type_matches'] += type_matches
    overall_counts['dir_matches'] += dir_matches
    overall_counts['mag_matches'] += mag_matches
    overall_counts['all_matches'] += all_matches

# Save Excel report
excel_path = f'{output_dir}/cluster_action_recommendations.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for cid in test_cluster_ids:
        if cid not in excel_rows_by_cluster:
            continue
        excel_rows_by_cluster[cid].to_excel(writer, sheet_name=f'cluster_{cid}', index=False)

# Save summary
summary_df = pd.DataFrame(all_episodes_summary)
summary_df.to_excel(f'{output_dir}/cluster_summary.xlsx', index=False)

# Print overall results
total = overall_counts['total_actual']
if total > 0:
    print(f'\n{"="*60}')
    print(f'OVERALL RESULTS (Alarm Clusters)')
    print(f'{"="*60}')
    print(f'Tag match:       {overall_counts["tag_matches"]}/{total} = {overall_counts["tag_matches"]/total*100:.1f}%')
    print(f'Type match:      {overall_counts["type_matches"]}/{total} = {overall_counts["type_matches"]/total*100:.1f}%')
    print(f'Direction match: {overall_counts["dir_matches"]}/{total} = {overall_counts["dir_matches"]/total*100:.1f}%')
    print(f'Magnitude match: {overall_counts["mag_matches"]}/{total} = {overall_counts["mag_matches"]/total*100:.1f}%')
    print(f'All-match:       {overall_counts["all_matches"]}/{total} = {overall_counts["all_matches"]/total*100:.1f}%')
    print(f'\n(All-match = tag + type + direction + magnitude all correct)')

print(f'\nResults saved to: {output_dir}/')
print(f'  - {len(excel_rows_by_cluster)} cluster visualizations (HTML)')
print(f'  - cluster_action_recommendations.xlsx')
print(f'  - cluster_summary.xlsx')

=== 4-Component Similarity + Sub-Minute Merging (Alarm Clusters) ===
Training lookup: 911 entries from 128 clusters
Test clusters: 47
Output: /home/h604827/ControlActions/RESULTS/similarity_test_results/alarm_clusters_v1/



Test clusters:   0%|          | 0/47 [00:00<?, ?it/s]

Test clusters: 100%|██████████| 47/47 [00:30<00:00,  1.52it/s]



OVERALL RESULTS (Alarm Clusters)
Tag match:       190/287 = 66.2%
Type match:      205/287 = 71.4%
Direction match: 127/287 = 44.3%
Magnitude match: 141/287 = 49.1%
All-match:       58/287 = 20.2%

(All-match = tag + type + direction + magnitude all correct)

Results saved to: /home/h604827/ControlActions/RESULTS/similarity_test_results/alarm_clusters_v1/
  - 47 cluster visualizations (HTML)
  - cluster_action_recommendations.xlsx
  - cluster_summary.xlsx
